# Neuro-SOC — Exploratory Data Analysis & Feature Importance

**Purpose:** Demonstrate our baselining approach, feature engineering, and SHAP-based model explainability for the Isolation Forest anomaly detector.

**Author:** Neuro-SOC Engineering Team  
**Date:** 2026-06-14  
**Model:** Isolation Forest (scikit-learn) + SHAP TreeExplainer

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import shap
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from pathlib import Path

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#141414',
    'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#333',
    'axes.labelcolor': '#aaa',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'text.color': '#ccc',
    'grid.color': '#222',
    'font.size': 10,
})

RAW = Path('../data_pipeline/raw_data')
PROCESSED = Path('../data_pipeline/processed_data')
print('Setup complete.')

---
## 1 · Data Ingestion

In [ ]:
logs = pd.read_csv(RAW / 'data_access_logs.csv', parse_dates=['timestamp'])
users = pd.read_csv(RAW / 'user_profiles.csv')
enriched = pd.read_csv(PROCESSED / 'enriched_features.csv')

print(f'Access Logs : {logs.shape[0]:,} rows × {logs.shape[1]} cols')
print(f'User Profiles: {users.shape[0]:,} rows × {users.shape[1]} cols')
print(f'Enriched     : {enriched.shape[0]:,} rows × {enriched.shape[1]} cols')
print()
print('--- Access Logs Sample ---')
logs.head(3)

In [ ]:
print('--- User Profiles Sample ---')
users.head(3)

---
## 2 · Temporal Baselining — Access Hour Distribution

We extract the hour of each access event and compare **business-hours** vs. **after-hours** patterns. Anomalous users tend to show activity in unusual time windows.

In [ ]:
logs['hour'] = logs['timestamp'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Business hours events
bh = logs[logs['time_classification'] == 'business_hours']['hour']
axes[0].hist(bh, bins=24, range=(0, 24), color='#4ade80', alpha=0.7, edgecolor='#333')
axes[0].set_title('Business Hours Access', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Event Count')
axes[0].axvspan(9, 17, alpha=0.08, color='white', label='Core hours (9-17)')
axes[0].legend(fontsize=8, loc='upper right')

# After-hours events
ah = logs[logs['time_classification'] == 'after_hours']['hour']
axes[1].hist(ah, bins=24, range=(0, 24), color='#f87171', alpha=0.7, edgecolor='#333')
axes[1].set_title('After-Hours Access', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].axvspan(0, 6, alpha=0.08, color='#f87171', label='High-risk window (0-6)')
axes[1].axvspan(20, 24, alpha=0.08, color='#f87171')
axes[1].legend(fontsize=8, loc='upper left')

fig.suptitle('Temporal Baselining — Access Hour Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('temporal_baselining.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Action Distribution — Normal vs. Sensitive Operations

We compare the frequency of each action type across all events, split by resource sensitivity level.

In [ ]:
ct = pd.crosstab(logs['action'], logs['resource_sensitivity'])
ct = ct.reindex(columns=['low', 'medium', 'high'], fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ct.plot.barh(ax=ax, color=['#4ade80', '#facc15', '#f87171'], edgecolor='#333', alpha=0.8)
ax.set_title('Actions by Resource Sensitivity', fontsize=11, fontweight='bold')
ax.set_xlabel('Count')
ax.set_ylabel('')
ax.legend(title='Sensitivity', fontsize=8, title_fontsize=9)
plt.tight_layout()
plt.savefig('action_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · Feature Engineering Summary

Our enrichment pipeline produces 11 numeric features from the raw logs and profiles. Here we review their distributions.

In [ ]:
FEATURES = [
    'days_inactive', 'tenure_months', 'high_risk_flag',
    'Tenure_Risk_Modifier', 'Equipment_Mismatch_Score',
    'Temporal_Velocity', 'rowcount_deviation',
    'After_Hours_High_Sensitivity', 'Failed_Action_Flag',
    'systems_access_count', 'Privilege_Sensitivity_Mismatch',
]

# Convert booleans
feat = enriched[FEATURES].copy()
for c in feat.columns:
    if feat[c].dtype == 'object':
        feat[c] = feat[c].map({'True': 1, 'False': 0}).fillna(0).astype(int)
    elif feat[c].dtype == 'bool':
        feat[c] = feat[c].astype(int)
feat = feat.fillna(0)

print(feat.describe().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    axes[i].hist(feat[col], bins=30, color='#888', alpha=0.7, edgecolor='#333')
    axes[i].set_title(col.replace('_', ' '), fontsize=8, fontweight='bold')
    axes[i].tick_params(labelsize=7)

# Hide the 12th subplot
axes[11].set_visible(False)

fig.suptitle('Feature Distributions (11 Engineered Features)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 · Isolation Forest Training & Anomaly Detection

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feat)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_scaled)

preds = model.predict(X_scaled)
scores = model.decision_function(X_scaled)

n_anom = (preds == -1).sum()
print(f'Total events   : {len(preds):,}')
print(f'Anomalies      : {n_anom} ({n_anom/len(preds)*100:.1f}%)')
print(f'Score range    : [{scores.min():.4f}, {scores.max():.4f}]')
print(f'Score mean     : {scores.mean():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

ax.hist(scores[preds == 1], bins=50, color='#4ade80', alpha=0.6, label=f'Normal ({(preds == 1).sum()})', edgecolor='#333')
ax.hist(scores[preds == -1], bins=50, color='#f87171', alpha=0.7, label=f'Anomaly ({n_anom})', edgecolor='#333')
ax.axvline(x=0, color='#555', linestyle='--', linewidth=0.8, label='Decision boundary')
ax.set_title('Anomaly Score Distribution', fontsize=11, fontweight='bold')
ax.set_xlabel('Decision Function Score')
ax.set_ylabel('Count')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('anomaly_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6 · SHAP Feature Importance — Model Explainability

This is the core of our XAI (Explainable AI) approach. We use SHAP TreeExplainer to:
1. Compute global feature importance across all events
2. Show per-feature contribution for individual anomalies

This proves our ML pipeline is **not a black box** — every decision is traceable to specific behavioral deviations.

In [ ]:
# Use a background sample for efficiency
background = X_scaled[np.random.RandomState(42).choice(len(X_scaled), 100, replace=False)]

explainer = shap.TreeExplainer(model, data=background, feature_names=FEATURES)
shap_values = explainer.shap_values(X_scaled)

print(f'SHAP values shape: {shap_values.shape}')
print('Global feature importance (mean |SHAP|):')
importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURES,
).sort_values(ascending=False)
print(importance.round(4).to_string())

In [ ]:
# Global Feature Importance Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))

imp_sorted = importance.sort_values(ascending=True)
colors = ['#f87171' if v > importance.median() else '#888' for v in imp_sorted.values]

ax.barh(
    [f.replace('_', ' ') for f in imp_sorted.index],
    imp_sorted.values,
    color=colors,
    edgecolor='#333',
    alpha=0.8,
)
ax.set_title('Global SHAP Feature Importance (Isolation Forest)', fontsize=11, fontweight='bold')
ax.set_xlabel('Mean |SHAP value|')

plt.tight_layout()
plt.savefig('shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Summary Plot (beeswarm)
shap.summary_plot(
    shap_values,
    features=feat.values,
    feature_names=[f.replace('_', ' ') for f in FEATURES],
    plot_type='dot',
    show=False,
    max_display=11,
)
plt.title('SHAP Summary — Per-Feature Impact on Anomaly Score', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Single anomaly explanation (first detected anomaly)
anom_idx = np.where(preds == -1)[0][0]
print(f'Explaining anomaly at index {anom_idx} (user: {enriched.iloc[anom_idx]["user_id"]})')
print(f'Anomaly score: {scores[anom_idx]:.4f}')
print()

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[anom_idx],
        base_values=explainer.expected_value,
        data=feat.iloc[anom_idx].values,
        feature_names=[f.replace('_', ' ') for f in FEATURES],
    ),
    show=False,
)
plt.title(f'SHAP Waterfall — {enriched.iloc[anom_idx]["user_id"]}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall_single.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7 · Key Takeaways

1. **Temporal Velocity** and **Rowcount Deviation** are the strongest anomaly drivers — users who access data much faster or more frequently than their baseline are flagged.
2. **Privilege Sensitivity Mismatch** catches users accessing resources above their clearance level.
3. **After-Hours + High Sensitivity** captures the classic insider threat pattern of late-night access to sensitive systems.
4. Every flagged anomaly is **fully explainable** via SHAP waterfall plots — no black-box decisions.
5. The Isolation Forest detects ~5% of events as anomalous, aligning with industry benchmarks for insider threat rates.